
### 1. General Exploration of the Dataset


In [ ]:
# Install seaborn.
%pip install seaborn

In [ ]:
%pip install folium

In [ ]:
# Import pandas, numpy, matplotlib, and seaborn.
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import matplotlib.image as mpimg 
import folium
from folium.plugins import HeatMap

cows = pd.read_csv("raw_data.csv") # Read in the data, allocating it to an object called 'cows'.

np.random.seed(0) # Set seed for reproducibility.

In [ ]:
print(cows.shape) # Dimensions of the data - contains 264742 rows and 18 columns.
cows # Look at the first and last 5 rows of the data (missing values can be observed).

In [ ]:
# Renames columns to remove spaces and hashtags.
# Creates a new copy of the specified columns with a different name.
cows["Pasture_ID"] = cows["Pasture ID"]
cows["Pasture_Version"] = cows["Pasture Version"]
cows["Message_Type"] = cows["Message Type"]
cows["Battery_Voltage"] = cows["Battery Voltage"]
cows["GSM_RAT"] = cows["GSM RAT"]
cows["GSM_Operator"] = cows["GSM Operator"]
cows["Position_Time"] = cows["Position Time"]
cows["Horizontal_Accuracy"] = cows["Horizontal Accuracy"]
cows["n_Satellites"] = cows["#Satellites"]
cows["Fence_Status"] = cows["Fence Status"]
cows["Collar_Status"] = cows["Collar Status"]

# Removes the originally named column so only the renamed column remains.
cows = cows.drop('Pasture ID', axis=1)
cows = cows.drop('Pasture Version', axis=1)
cows = cows.drop('Message Type', axis=1)
cows = cows.drop('Battery Voltage', axis=1)
cows = cows.drop('GSM RAT', axis=1)
cows = cows.drop('GSM Operator', axis=1)
cows = cows.drop('Position Time', axis=1)
cows = cows.drop('Horizontal Accuracy', axis=1)
cows = cows.drop('#Satellites', axis=1)
cows = cows.drop('Fence Status', axis=1)
cows = cows.drop('Collar Status', axis=1)

In [ ]:
cows.dtypes # Shows the data types contained within each field.

In [ ]:
# Number of missing data points per field in descending order including only the 10 fields with the most missing data.
mvcount = cows.isnull().sum().sort_values(ascending = False).head(10)
mvcount

In [ ]:
# Calculates % missing data per field in descending order including only the 10 fields with the most missing data.
proportionm = cows.isna().mean().sort_values(ascending = False).head(10)
percentm = 100 * proportionm
percentm

In [ ]:
print(cows.nunique().sort_values(ascending = True).head(25)) # Number of unique values in each field in descending order.

In [ ]:
chunk = cows.loc[cows['Pasture_Version'] == 47]

In [ ]:
chunk.to_csv('cows_filtered1.csv') # Saves datafrme as a CSV file.


### 3. Statistics and Visualisations


In [ ]:
# Converts column to datetime (has to be done prior to statistics and visualisations).
cows['Time'] = pd.to_datetime(cows['Time'])
cows['Time'] = (cows['Time'] - pd.Timestamp('2024-02-09 00:31:27+00:00')) / pd.Timedelta(days=1)
cows['Time']

In [ ]:
# Converts column to datetime (has to be done prior to statistics and visualisations).
cows['Position_Time'] = pd.to_datetime(cows['Position_Time'])
cows['Position_Time'] = (cows['Position_Time'] - pd.Timestamp('2024-02-09 00:31:27+00')) / pd.Timedelta(days=1)
cows['Position_Time']

In [ ]:
# Assigns all numeric variables to an object.
continuous1 = ["Battery_Voltage", "n_Satellites", "Horizontal_Accuracy", "HDOP", "Latitude", "Longitude", "Time", "Position_Time"]
# "Activity" excluded in this case due to missing values (100% missing).

# Assigns categorical variables to an object.
categorical1 = ["GSM_RAT", "GSM_Operator", "Mode", "Fence_Status", "Collar_Status", "Pasture_ID", "Serial", "Pasture_Version"]
# "Message_Type" excluded in this case due to only one class ("poll").


### 3.1 Numeric Variables


In [ ]:
#Calculates descriptive statistics for numeric variables.
numeric1 = cows[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = cows[column].median()
print(numeric1)

In [ ]:
# Histograms showing the distribution for each numerical variable.
for i, col in enumerate(continuous1):
    plt.subplot(7, 2, i + 1)
    cows[col].hist(bins=30, figsize=(15, 25), edgecolor='white')
    plt.title(f'Count Plot of {col}')
plt.suptitle('Figure 2. Histograms of Continuous Features', y=1.02)
plt.tight_layout()
plt.show()


### 3.2. Categorical Variables


In [ ]:
# Class means.
mean_price_by = cows.groupby('GSM_RAT')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('GSM_Operator')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Mode')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Fence_Status')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Collar_Status')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Serial')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

In [ ]:
# Count Plots showing the distribution of categories for each numerical variable.
plt.figure(figsize=(15, 25))
for i, col in enumerate(categorical1):
    plt.subplot(5, 2, i + 1)
    sns.countplot(y=cows[col])
    plt.title(f'Count Plot of {col}')
plt.suptitle('Figure 3. Count Plots of Categorical Features', y=1.02)
plt.tight_layout()
plt.show()


### 3.3 Inspection for further Issues


In [ ]:
# Shows the frequency of individual values within a field.
cows.GSM_RAT.value_counts()
# Applied to variables to look for further data reliability issues such as outliers.

In [ ]:
# Shows the frequency of individual values within a field.
cows.GSM_Operator.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Mode.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Fence_Status.value_counts()

# Filter by these.

In [ ]:
# Shows the frequency of individual values within a field.
cows.Collar_Status.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Pasture_ID.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Serial.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Pasture_Version.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Battery_Voltage.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.n_Satellites.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Horizontal_Accuracy.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.HDOP.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Latitude.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Longitude.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Time.value_counts()

In [ ]:
# Shows the frequency of individual values within a field.
cows.Position_Time.value_counts()


### 4. Relationships



### 4.1 Numeric Variables


In [ ]:
corr_matrix = cows.corr(numeric_only=True)

nfeatures = ["Battery_Voltage", "n_Satellites", "Horizontal_Accuracy", "HDOP", "Latitude", "Longitude", "Time", "Position_Time"]

plt.figure(figsize=(15, 10))
sns.heatmap(cows[nfeatures].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Figure 9. Correlation Matrix')
plt.show()

In [ ]:
other_features = ["Battery_Voltage", "n_Satellites", "Horizontal_Accuracy", "HDOP", "Latitude", "Longitude", "Time"]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(other_features):
    sns.scatterplot(data=cows, x=feature, y='Position_Time', ax=axes[i])
    axes[i].set_title(f'{feature} vs Position_Time')
plt.suptitle('Figure 5. Scatterplots of Numerical Features', y=1.02)
plt.tight_layout()
plt.show()


### 4.2 Categorical Variables


In [ ]:
# Boxplots and Violin plots to show time related differences in category classes.
# ["GSM_RAT", "GSM_Operator", "Mode", "Fence_Status", "Collar_Status", "Pasture_ID", "Serial", "Pasture_Version"]
categorical_features = ["Mode", "Fence_Status", "Collar_Status"]
plt.figure(figsize=(15, 10))
for i, col in enumerate(categorical_features):
    plt.subplot(3, 2, i + 1)
    sns.boxplot(x='Position_Time', y=col, data=cows)
    plt.title(f'{col} Distribution by Position_Time')
plt.suptitle('Figure 7. Box Plots of Categorical Features', y=1.02)
plt.tight_layout()
plt.show()

v_features = ["Mode", "Fence_Status", "Collar_Status"]
plt.figure(figsize=(15, 10))
for i, col in enumerate(v_features):
    plt.subplot(3, 2, i + 1)
    sns.violinplot(x='Position_Time', y=col, data=cows)
    plt.title(f'{col} Distribution by Position_Time')
plt.suptitle('Figure 8. Violin Plots of Categorical Features', y=1.02)
plt.tight_layout()
plt.show()


### 5. Geospacial Distribution


In [ ]:
#exactcows = cows.loc[cows['is_location_exact'] == 't']

plt.figure(figsize=(12, 10))

sns.scatterplot(
    x='Longitude',
    y='Latitude',
    hue='Position_Time', # Could try and add stuff like step length or turning angle here.
    data=cows,
    palette='coolwarm',
    alpha=0.6,
    edgecolor=None)

plt.title('Figure 12. Geospatial Distribution of Cattle Movement Coloured by Position Time')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Position Time')
plt.show()

In [ ]:
import folium
from folium.plugins import HeatMap

# 1. Filter for exact locations (uncomment if you want to use the filter)
# exact_cows = cows.loc[cows['is_location_exact'] == 't']
# geo_data = exact_cows[['Latitude', 'Longitude', 'Position_Time']].dropna()

# 2. Extract and clean coordinate data
geo_data = cows[["Latitude", "Longitude"]].dropna()

# 3. Format data for the heatmap (Lat, Long, optional weight)
# Note: If you don't need time-weighting, Folium automatically treats each row as a point of equal weight.
heat_data = geo_data[["Latitude", "Longitude"]].values.tolist()

# 4. Generate the map object
map_center = [geo_data["Latitude"].mean(), geo_data["Longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=11)

# 5. Add the heatmap layer
HeatMap(heat_data, radius=12, blur=10, max_zoom=12).add_to(m)

# 6. SAVE THE INTERACTIVE MAP
m.save("cow_heatmap.html")

m


### 6. Data Transformations


In [ ]:
# Converts all values over 200 to a float nan (>200 was used to select outliers after inspecting the 'age' histogram and .unique() outputs).
cows.loc[cows['Latitude'] > 52.645, 'Latitude'] = np.nan
cows.loc[cows['Latitude'] < 52.620, 'Latitude'] = np.nan
cows.loc[cows['Longitude'] > 1.28, 'Longitude'] = np.nan
cows.loc[cows['Longitude'] < 1.24, 'Longitude'] = np.nan

In [ ]:
# Removes 'Activity' column.
cows = cows.drop('Activity', axis=1)

In [ ]:
# Calculates % missing data per field in descending order including only the 10 fields with the most missing data.
proportionm = cows.isna().mean().sort_values(ascending = False).head(10)
percentm = 100 * proportionm
percentm
# Latitude and Longitude have increased as outliers were removed.

In [ ]:
# Removes the remaining rows with missing values for the respective fields.
cows.dropna(inplace=True)

In [ ]:
# Converts data type from float to integer (discrete).
cows["Pasture_ID"]=cows["Pasture_ID"].astype(int)
cows["Pasture_Version"]=cows["Pasture_Version"].astype(int)
cows["n_Satellites"]=cows["n_Satellites"].astype(int)
print(cows['Pasture_ID'])
print(cows['Pasture_Version'])
print(cows['n_Satellites'])



# geopy.distance.geodesic(coord1, coord2).meters


### 7. Feature Engineering


In [ ]:
# Creates a new variable showing the percentage of the average store visit time that a customer spends queueing.
#cows[''] = cows['']

#shows the top 5 rows of the new features.
#display(cows[['']].head())


### 8. Post-Transformation Outputs


In [ ]:
print(cows.shape) # Dimensions of the data - contains 262429 rows and 17 columns.
cows # Look at the first and last 5 rows of the data (missing values can be observed).

In [ ]:
cows.dtypes # Shows the data types contained within each field.

In [ ]:
# Number of missing data points per field in descending order including only the 10 fields with the most missing data.
mvcount = cows.isnull().sum().sort_values(ascending = False).head(10)
mvcount

In [ ]:
print(cows.nunique().sort_values(ascending = True).head(25)) # Number of unique values in each field in descending order.


### 9. Post-Transformation Statistics and Visualisations


In [ ]:
# Assigns all transfrormed numeric variables to an object.
continuous2 = ["Battery_Voltage", "n_Satellites", "Horizontal_Accuracy", "HDOP", "Latitude", "Longitude", "Time", "Position_Time"]

# Assigns transfrormed categorical variables to an object.
categorical2 = ["GSM_RAT", "GSM_Operator", "Mode", "Fence_Status", "Collar_Status", "Serial"]


### 9.1 Numeric Variables


In [ ]:
#Calculates descriptive statistics for numeric variables.
numeric2 = cows[continuous2].describe()
for column in continuous2:
    numeric2.loc['median',column] = cows[column].median()
print(numeric2)

In [ ]:
# Histograms showing the distribution for each numerical variable.
for i, col in enumerate(continuous2):
    plt.subplot(8, 2, i + 1)
    cows[col].hist(bins=30, figsize=(15, 25), edgecolor='white')
    plt.title(f'Count Plot of {col}')
plt.suptitle('Figure 2. Histograms of New Continuous Features', y=1.02)
plt.tight_layout()
plt.show()


### 9.2 Categorical Variables


In [ ]:
# Class means.
mean_price_by = cows.groupby('GSM_RAT')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('GSM_Operator')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Mode')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Fence_Status')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Collar_Status')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

mean_price_by = cows.groupby('Serial')['Position_Time'].mean().sort_values(ascending=False)
print("Mean Position_Time by :\n", mean_price_by)
print("\n" + "-"*30 + "\n")

In [ ]:
# Count Plots showing the distribution of categories for each numerical variable.
plt.figure(figsize=(15, 25))
for i, col in enumerate(categorical2):
    plt.subplot(5, 2, i + 1)
    sns.countplot(y=cows[col])
    plt.title(f'Count Plot of {col}')
plt.suptitle('Figure 3. Count Plots of Categorical Features', y=1.02)
plt.tight_layout()
plt.show()


### 10. Post-Transformantion Relationships



### 10.1 Numeric Variables


In [ ]:
corr_matrix = cows.corr(numeric_only=True)

nfeatures = ["Battery_Voltage", "n_Satellites", "Horizontal_Accuracy", "HDOP", "Latitude", "Longitude", "Time", "Position_Time"]

plt.figure(figsize=(15, 10))
sns.heatmap(cows[nfeatures].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Figure 9. Correlation Matrix')
plt.show()

#price_correlation = corr_matrix['price'].sort_values(ascending=False)
#print("Correlation with Price:\n", price_correlation)

In [ ]:
other_features = ["Battery_Voltage", "n_Satellites", "Horizontal_Accuracy", "HDOP", "Latitude", "Longitude", "Time", "Position_Time"]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(other_features):
    sns.scatterplot(data=cows, x=feature, y='Position_Time', ax=axes[i])
    axes[i].set_title(f'{feature} vs Position_Time')
plt.suptitle('Figure 5. Scatterplots of Numerical Features', y=1.02)
plt.tight_layout()
plt.show()


### 9.2 Categorical Variables


In [ ]:
# Boxplots and Violin plots to show time related differences in category classes.
# ["GSM_RAT", "GSM_Operator", "Mode", "Fence_Status", "Collar_Status", "Pasture_ID", "Serial", "Pasture_Version"]
categorical_features = ["Mode", "Fence_Status", "Collar_Status"]
plt.figure(figsize=(15, 10))
for i, col in enumerate(categorical_features):
    plt.subplot(3, 2, i + 1)
    sns.boxplot(x='Position_Time', y=col, data=cows)
    plt.title(f'{col} Distribution by Position_Time')
plt.suptitle('Figure 7. Box Plots of Categorical Features', y=1.02)
plt.tight_layout()
plt.show()

v_features = ["Mode", "Fence_Status", "Collar_Status"]
plt.figure(figsize=(15, 10))
for i, col in enumerate(v_features):
    plt.subplot(3, 2, i + 1)
    sns.violinplot(x='Position_Time', y=col, data=cows)
    plt.title(f'{col} Distribution by Position_Time')
plt.suptitle('Figure 8. Violin Plots of Categorical Features', y=1.02)
plt.tight_layout()
plt.show()


### 11. Post-Transformation Geospacial Distribution


In [ ]:

#filtered = cows.loc[cows['Pasture_Version'] == 47]
filtered = cows.loc[cows['Pasture_ID'] == 55540]
filtered2 = filtered.loc[filtered['Pasture_Version'] == 47]
filtered3 = filtered2.loc[filtered2['Serial'] == 294366]
filtered4 = filtered3.loc[filtered3['Position_Time'] > 392]
filtered5 = filtered2.loc[filtered2['Position_Time'] < 404]


plt.figure(figsize=(12, 10))

sns.scatterplot(
    x='Longitude',
    y='Latitude',
    hue='Position_Time', # Could try and add stuff like step length or turning angle here.
    style='Serial',
    #data=cows,
    #data=filtered,
    data=filtered2,
    #data=filtered3,
    #data=filtered4,
    #data=filtered5,
    palette='coolwarm',
    alpha=0.6,
    edgecolor=None)

plt.title('Figure 12. Geospatial Distribution of Cattle Movement Coloured by Position Time')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Position Time')
plt.show()

In [ ]:
#exactcows = cows.loc[cows['is_location_exact'] == 't']

geo_data = cows[['Latitude', 'Longitude', 'Position_Time']].dropna() # Can change "cows to vairious filters from last markdown cell.

geo_data['weight'] = geo_data['Position_Time'] / geo_data['Position_Time'].max()

heat_data = geo_data[['Latitude', 'Longitude', 'weight']].values.tolist()

map_center = [geo_data['Latitude'].mean(), geo_data['Longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=11)

HeatMap(heat_data, radius=15, blur=20, max_zoom=12).add_to(m)

m